# How to Use the CW305 / CW310 / CW340 Standalone

Our large FPGA target boards are designed to work with our [capture
hardware](../../Capture/index.md), but they can also easily be used
"standalone", without any capture hardware.

Perhaps you have a fancy
expensive oscilloscope and want to use it to capture traces
([asynchronously](../common_questions/synchronous_sampling)!).

This note teaches how.

**Supported Capture Hardware:**\
None needed!

**Supported Targets:**\
✅ CW305\
✅ CW310\
✅ CW340\
❌ CW312T-XC7A35\
❌ CW312T-iCE40

**Required ChipWhisperer software installation:**\
✅ any release

The main requirements we need to fill for standalone use are:
1. Clocking the target.
2. Talking to the target.
3. Triggering the capture.
4. Getting power traces.

We'll show you how here. We'll assume a CW305 target, but all the principles apply equally to the CW310/CW340.

## Step 0: Connect

Here we use our usual AES example bitstream.

If you want to use a custom bitstream instead, use: `target = cw.target(None, cw.targets.CW305, force=True, bsfile='/path/to/bitstream', platform='cw305')`

In [2]:
import chipwhisperer as cw
target = cw.target(None, cw.targets.CW305, force=True, fpga_id='100t', platform='cw305')

## Step 1: Clocks

We program the CW305's onboard PLL to generate a 10 MHz clock. The PLL1 clock is connected to FPGA pin N13 ([documentation](../../Targets/CW305%20Artix%20FPGA.md#pll)); in the the [AES reference design](https://github.com/newaetech/chipwhisperer/blob/develop/firmware/fpgas/aes/vivado/cw305.xdc) this goes to the `pll_clk1` input.

In [3]:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)
target.pll.pll_outfreq_set(10E6, 1)

## Step 2: Communication

With our CW305 example designs, communication is done over the USB interface; it does not rely on a ChipWhisperer scope.

So, we can read and write FPGA registers the same way:

In [15]:
target.fpga_buildtime

'10/19/2023, 12:04'

In [16]:
target.fpga_write(target.REG_CRYPT_KEY, range(8))

In [17]:
assert list(target.fpga_read(target.REG_CRYPT_KEY, 8)) == list(range(8))

## Step 3: Triggering

By "triggering", we mean: (a) making the target "go", and (b) having the target raise a trigger line to indicate that a trace capture should begin.

(a) is just an instance of target communication, and, as explained in step (2), this happens over the USB interface, so no changes are needed here.

(b) this is up to you but in most of our examples, the TIO4 line is used as a trigger line. On the CW305, this is accessible on the 20-pin header port and on the "TRIG" test point (labeled TP1, near the bottom right corner of the board.

The CW305.py `go()` method does this for us. When you run this, you should see the TIO4 line pulse high for 11 cycles of the target's 10 MHz clock (because that's how long the encryption takes):

In [26]:
target.go()

## Step 4: Getting Power Traces

This part depends on your oscilloscope, so here you're on your own.